In [ ]:
# ===== Image -> 3D (Hunyuan3D-2mini + minimal Gradio viewer) : single file, Kaggle T4 =====
import os, sys, subprocess, types
import importlib.metadata as md

def sh(cmd, check=True):
    subprocess.run(cmd, shell=True, check=check)

# ---------- 1. Setup (numpy/torch locked, never upgraded) ----------
if not os.path.exists("Hunyuan3D-2"):
    sh("git clone https://github.com/Tencent/Hunyuan3D-2.git")

with open("/tmp/constraints.txt", "w") as f:
    f.write(f"numpy=={md.version('numpy')}\n")

sh("pip install -q -c /tmp/constraints.txt diffusers accelerate safetensors einops omegaconf "
   "transformers trimesh rembg onnxruntime gradio pyyaml tqdm")
sh("pip install -q -c /tmp/constraints.txt pymeshlab", check=False)   # optional, stubbed below if it fails
sys.path.insert(0, os.path.abspath("Hunyuan3D-2"))

try:
    import pymeshlab  # noqa
except Exception:
    sys.modules["pymeshlab"] = types.ModuleType("pymeshlab")   # only used for mesh reduction, which we skip

# ---------- 2. Imports ----------
import tempfile
from itertools import permutations, product
import numpy as np
import torch
import trimesh
import rembg
import gradio as gr
from PIL import Image
from scipy import ndimage
from hy3dgen.shapegen import Hunyuan3DDiTFlowMatchingPipeline

# ---------- 3. Settings ----------
STEPS = 50          # 30 = faster, 50 = better
OCTREE = 384        # mesh detail: 256 / 320 / 384 / 512 (higher = sharper, bigger file, more VRAM)
SEED = 2025

# ---------- 4. Compatibility patch: old DINOv2 checkpoint keys -> new transformers names ----------
_orig_load_state_dict = torch.nn.Module.load_state_dict

def _remap_dinov2_keys(sd):
    out = {}
    for k, v in sd.items():
        if ".attention.attention.query." in k:
            out[k.replace(".attention.attention.query.", ".attention.q_proj.")] = v
        elif ".attention.attention.key." in k:
            out[k.replace(".attention.attention.key.", ".attention.k_proj.")] = v
        elif ".attention.attention.value." in k:
            out[k.replace(".attention.attention.value.", ".attention.v_proj.")] = v
        elif ".attention.output.dense." in k:
            out[k.replace(".attention.output.dense.", ".attention.o_proj.")] = v
        elif ".mlp.weights_in." in k:
            gate, up = v.chunk(2, dim=0)                 # SwiGLU: first half = gate, second half = up
            out[k.replace(".mlp.weights_in.", ".mlp.gate_proj.")] = gate.contiguous()
            out[k.replace(".mlp.weights_in.", ".mlp.up_proj.")] = up.contiguous()
        elif ".mlp.weights_out." in k:
            out[k.replace(".mlp.weights_out.", ".mlp.down_proj.")] = v
        else:
            out[k] = v
    return out

def _patched_load_state_dict(self, state_dict, *args, **kwargs):
    try:
        model_wants_new = any(".attention.q_proj." in n for n in self.state_dict().keys())
        ckpt_has_old = any(".attention.attention.query." in n for n in state_dict.keys())
        if model_wants_new and ckpt_has_old:
            state_dict = _remap_dinov2_keys(state_dict)
            print("Remapped DINOv2 checkpoint keys for newer transformers")
    except Exception as e:
        print("key remap skipped:", e)
    return _orig_load_state_dict(self, state_dict, *args, **kwargs)

torch.nn.Module.load_state_dict = _patched_load_state_dict

# ---------- 5. Load models ----------
device = "cuda" if torch.cuda.is_available() else "cpu"
pipe = Hunyuan3DDiTFlowMatchingPipeline.from_pretrained(
    "tencent/Hunyuan3D-2mini",
    subfolder="hunyuan3d-dit-v2-mini",
    use_safetensors=True,
    device=device,
)
try:
    rembg_session = rembg.new_session("isnet-general-use")   # better cut-outs for products
except Exception:
    rembg_session = rembg.new_session()
print("Loaded on", device)

# ---------- 6. Helpers ----------
def prep(img):
    """Remove background, crop to object, pad to a 512x512 square RGBA."""
    rgba = rembg.remove(img.convert("RGB"), session=rembg_session).convert("RGBA")
    a = np.array(rgba)
    ys, xs = np.where(a[..., 3] > 10)
    if len(xs) == 0:
        raise gr.Error("No object found in the image.")
    a = a[ys.min():ys.max() + 1, xs.min():xs.max() + 1]
    h, w = a.shape[:2]
    side = int(max(h, w) * 1.15)
    canvas = np.zeros((side, side, 4), np.uint8)
    y0, x0 = (side - h) // 2, (side - w) // 2
    canvas[y0:y0 + h, x0:x0 + w] = a
    return Image.fromarray(canvas).resize((512, 512), Image.LANCZOS)

def keep_main(mesh):
    """Drop floating fragments (keep pieces >= 5% of the largest one)."""
    try:
        parts = mesh.split(only_watertight=False)
        if len(parts) > 1:
            big = max(len(p.faces) for p in parts)
            parts = [p for p in parts if len(p.faces) >= 0.05 * big]
            mesh = trimesh.util.concatenate(parts)
    except Exception as e:
        print("split skipped:", e)
    return mesh

def rotations():
    for perm in permutations(range(3)):
        for sg in product([1, -1], repeat=3):
            M = np.zeros((3, 3))
            for i in range(3):
                M[i, perm[i]] = sg[i]
            if np.linalg.det(M) > 0:
                yield M

def colorize_and_orient(mesh, rgba, G=128):
    """Find which mesh orientation matches the photo silhouette, rotate the mesh upright
    and facing the camera, then project the photo colors onto it."""
    V = np.asarray(mesh.vertices, dtype=np.float64)
    V = V - (V.min(0) + V.max(0)) / 2
    N = np.asarray(mesh.vertex_normals, dtype=np.float64)

    arr = np.array(rgba)
    mask = arr[..., 3] > 127
    rgb = arr[..., :3]
    ys, xs = np.where(mask)
    mw, mh = xs.max() - xs.min() + 1, ys.max() - ys.min() + 1
    mcx, mcy = (xs.max() + xs.min()) / 2, (ys.max() + ys.min()) / 2
    mask_small = np.array(Image.fromarray((mask * 255).astype(np.uint8)).resize((G, G))) > 127

    def project(P):
        u, v = P[:, 0], -P[:, 1]
        s = (mw + mh) / (np.ptp(u) + np.ptp(v) + 1e-9)
        px = mcx + s * (u - (u.max() + u.min()) / 2)
        py = mcy + s * (v - (v.max() + v.min()) / 2)
        return px, py

    best_M, best_iou = None, -1
    Vs = V[::4]
    for M in rotations():
        px, py = project(Vs @ M.T)
        gx = np.clip((px * G / 512).astype(int), 0, G - 1)
        gy = np.clip((py * G / 512).astype(int), 0, G - 1)
        occ = np.zeros((G, G), bool)
        occ[gy, gx] = True
        occ = ndimage.binary_fill_holes(ndimage.binary_closing(occ, iterations=2))
        iou = (occ & mask_small).sum() / max((occ | mask_small).sum(), 1)
        if iou > best_iou:
            best_iou, best_M = iou, M
    print(f"Orientation match IoU: {best_iou:.2f}")

    P = V @ best_M.T
    px, py = project(P)

    # fill colors outside the (slightly eroded) mask with nearest inside color, avoids edge halos
    inner = ndimage.binary_erosion(mask, iterations=3)
    if inner.sum() < 100:
        inner = mask
    idx = ndimage.distance_transform_edt(~inner, return_distances=False, return_indices=True)
    rgb_f = rgb[idx[0], idx[1]]

    xi = np.clip(px.astype(int), 0, 511)
    yi = np.clip(py.astype(int), 0, 511)
    col = rgb_f[yi, xi].astype(np.float32)
    facing = (N @ best_M.T)[:, 2]
    col[facing < 0] *= 0.8                                   # back side: mirrored photo, slightly darker
    colors = np.concatenate([col, np.full((len(col), 1), 255, np.float32)], axis=1).astype(np.uint8)

    P = P / (np.abs(P).max() + 1e-9)
    return trimesh.Trimesh(vertices=P, faces=mesh.faces, vertex_colors=colors, process=False)

def generate(input_image):
    if input_image is None:
        return None, None, None
    rgba = prep(input_image)
    with torch.no_grad():
        mesh = pipe(
            image=rgba,
            num_inference_steps=STEPS,
            octree_resolution=OCTREE,
            num_chunks=20000,
            generator=torch.manual_seed(SEED),
            output_type="trimesh",
        )[0]
    mesh = keep_main(mesh)
    mesh = colorize_and_orient(mesh, rgba)
    out_path = tempfile.NamedTemporaryFile(suffix=".glb", delete=False).name
    mesh.export(out_path)
    if device == "cuda":
        torch.cuda.empty_cache()
    return out_path, out_path, rgba

# ---------- 7. Minimal UI ----------
CSS = """
<style>
  .gradio-container { max-width: 100% !important; padding: 8px !important; }
  #viewer { height: 92vh !important; }
  #viewer canvas { height: 100% !important; }
  footer { display: none !important; }
</style>
"""

with gr.Blocks(title="Image to 3D") as demo:
    gr.HTML(CSS)
    with gr.Row(equal_height=False):
        with gr.Column(scale=5):
            viewer = gr.Model3D(
                label="3D view (drag to rotate, scroll to zoom)",
                clear_color=(0.93, 0.93, 0.93, 1.0),
                height=800,
                elem_id="viewer",
            )
        with gr.Column(scale=1, min_width=240):
            in_img = gr.Image(label="Upload image (one object)", type="pil", image_mode="RGBA", height=240)
            proc = gr.Image(label="What the model sees", interactive=False, height=160)
            dl = gr.File(label="Download .glb", height=90)

    in_img.upload(generate, in_img, [viewer, dl, proc])
    in_img.clear(lambda: (None, None, None), None, [viewer, dl, proc])

demo.queue().launch(share=True, debug=True)

Cloning into 'Hunyuan3D-2'...
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dopamine-rl 4.1.2 requires gym<=0.25.2, but you have gym 0.26.2 which is incompatible.
/usr/local/lib/python3.13/dist-packages/numba/np/ufunc/parallel.py:371: NumbaWarning: The TBB threading layer requires TBB version 2021 update 6 or later i.e., TBB_INTERFACE_VERSION >= 12060. Found TBB_INTERFACE_VERSION = 12050. The TBB threading layer is disabled.
  warnings.warn(problem)
2026-11-12 07:53:14,223 - hy3dgen.shapgen - INFO - Try to load model from local path: /root/.cache/hy3dgen/tencent/Hunyuan3D-2mini/hunyuan3d-dit-v2-mini
2026-11-12 07:53:14,224 - hy3dgen.shapgen - INFO - Model path not exists, try to download from huggingface


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

2026-11-12 07:53:51,240 - hy3dgen.shapgen - INFO - Loading model from /root/.cache/huggingface/hub/models--tencent--Hunyuan3D-2mini/snapshots/f90a0f7df7d5e6f71109cf333f6a95a0ae3194a6/hunyuan3d-dit-v2-mini/model.fp16.safetensors


PointCrossAttentionEncoder INFO: pc_sharpedge_size is given, using pc_size=5120, pc_sharpedge_size=5120
Remapped DINOv2 checkpoint keys for newer transformers


  0%|                                               | 0.00/179M [00:00<?, ?B/s]

... (27 outputs left out)

Volume Decoding: 100%|██████████| 2854/2854 [01:26<00:00, 32.82it/s]


Orientation match IoU: 0.54


Volume Decoding: 100%|██████████| 2854/2854 [01:28<00:00, 32.29it/s]


Orientation match IoU: 0.86
